In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from SystHelper import *
from CommonTrain import *
from Model import Seq

2023-07-03 16:08:29.835921: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6fa5c/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-c1d8e/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow/contrib

Welcome to JupyROOT 6.28/00


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]

# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1] # ERA
REGION = sys.argv[2]
PART = sys.argv[3]
ERA = INPUTDIR

# # for dev
# VERSION = '20230527deepCSV-SR2'
# CHANNEL="ee"
# INPUTDIR = setting["INPUTDIR"]+'/em18/'
# ERA = "ee16"
# INPUTDIR = '/eos/user/h/hkwon/Run2/Preselection/20230526deepCSV/Presel/ee16/'
# # INPUTDIR = '/eos/user/h/hkwon/SWAN_projects/Zpeak/20230502pdf_17/Presel/ee17/'
# REGION = 'SR'
# PART = 'dummy'

20230430wp80_ANv3


# Load data

In [3]:
if REGION == 'SR':
    file = 'nested_dic_df_SR1.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_TTCR.pkl'
#     file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 
dic_df_met_ue_u_unsort=nested_dic_df['met_ue_u']
dic_df_met_ue_d_unsort=nested_dic_df['met_ue_d'] 

# # temp!!
for key in dic_df_met_u_unsort:
    df = dic_df_met_u_unsort[key]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_u_unsort[key] = df.drop(i)
    df = dic_df_met_d_unsort[key]
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_d_unsort[key] = df.drop(i)
        
selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
dic_df_met_ue_u={}
dic_df_met_ue_d={}
for key in dic_df:
#     print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]    
    dic_df_met_ue_u[key]=dic_df_met_ue_u_unsort[key][selected_columns]    
    dic_df_met_ue_d[key]=dic_df_met_ue_d_unsort[key][selected_columns]

In [4]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre',
       ...
       'weighteffSFUp', 'weighteffSFDown', 'weighttopSFUp', 'weighttopSFDown',
       'weightgenweightUp', 'weightgenweightDown', 'weighttrigSFUp',
       'weighttrigSFDown', 'weightbtagSFUp', 'weightbtagSFDown'],
      dtype='object', length=169)
Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_trail_pre', 'Zpt_pre',
       'dRll_pre', 'PuppiMET_pre', 'dPhilMET_pre', 'dPhillMET_pre', 'U_pre',
       'dPhill_pre', 'MT2_pre', 'JZB_pre'],
      dtype='object')
Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_trail_pre', 'Zpt_pre',
       'dRll_pre', 'PuppiMET_pre', 'dPhilMET_pre', 'dPhillMET_pre', 'U_pre',
       'dPhill_pre', 'MT2_pre', 'JZB_pre'],
      dtype='object')


In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [8]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            
    year = ERA.replace('mm', '')
    year = year.replace('ee', '')
    year = year.replace('em', '')

    #-- implement systematics --#   
    os.system("mkdir -p Outputs/DNN/"+REGION)
    os.system("mkdir -p Outputs/DNN/"+REGION+"/"+year)
    os.system("mkdir -p Outputs/DNN/"+REGION+"/sys")
    os.system("mkdir -p Outputs/DNN/"+REGION+"/root_files")
#     sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.6
    else:
        threshold = 1    
    
    # draw plot with syst
#     dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, dic_nested_shape_sys, sys_list, "score"+model_mass, r, "region_dummy")
    file_prefix = "Outputs/DNN/"+REGION
    sys_list = get_sys_list()
    dic_sys_sqr = get_dic_sys(model_mass, ERA, dic_predict_sort, dic_nested_shape_sys, sys_list, "score"+model_mass, r, REGION, threshold, file_prefix, file_str=VERSION)

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/DNN/"+REGION+"/"+year+"/"+model_mass+VERSION, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/DNN/"+REGION+"/"+model_mass+VERSION)     
    hist.export_yields_table(file_name="Outputs/DNN/"+REGION+"/"+model_mass)     
     
    return None

# Load model and evaluate

In [10]:
import time

nbin = 20
if REGION == "VR":
    nbin = 20
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
if PART=="one":
    model_masses = [m for m in dic_df_train if 'Zp-1' in m or 'Zp-2' in m]
if PART=="two":
    model_masses = [m for m in dic_df_train if 'Zp-3' in m or 'Zp-4' in m]
# model_masses = ['Zp-2500_CH-345', 'Zp-4100_CH-345']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = INPUTDIR

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d, nested_dic_predict_met_ue_u, nested_dic_predict_met_ue_d = [{} for _ in range(5)]

# masses loop
for model_mass in model_masses:
    
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]

#     file_postfix = 'test'

    dic_df_train_tmp, dic_df_met_u_tmp, dic_df_met_d_tmp, dic_df_met_ue_u_tmp, dic_df_met_ue_d_tmp = [{} for _ in range(5)]        
    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d, dic_df_scale_met_ue_u, dic_df_scale_met_ue_d = [{} for _ in range(5)]        
    
    
    for key2 in dic_df_train:
        dic_df_train_tmp[key2] = dic_df_train[key2].copy(deep=True)
        dic_df_met_u_tmp[key2] = dic_df_met_u[key2].copy(deep=True)
        dic_df_met_d_tmp[key2] = dic_df_met_d[key2].copy(deep=True)
        dic_df_met_ue_u_tmp[key2] = dic_df_met_ue_u[key2].copy(deep=True)
        dic_df_met_ue_d_tmp[key2] = dic_df_met_ue_d[key2].copy(deep=True)
        dic_df_train_tmp[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train_tmp[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))       
        dic_df_met_u_tmp[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u_tmp[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d_tmp[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d_tmp[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))        
        dic_df_met_ue_u_tmp[key2]['zp_m'] = np.full(dic_df_met_ue_u[key2].shape[0], int(zp_m))
        dic_df_met_ue_u_tmp[key2]['ch_m'] = np.full(dic_df_met_ue_u[key2].shape[0], int(ch_m))          
        dic_df_met_ue_d_tmp[key2]['zp_m'] = np.full(dic_df_met_ue_d[key2].shape[0], int(zp_m))
        dic_df_met_ue_d_tmp[key2]['ch_m'] = np.full(dic_df_met_ue_d[key2].shape[0], int(ch_m))       

        if dic_df_train[key2].empty:
            continue
        dic_df_scale[key2] = Transformer.transform(dic_df_train_tmp[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u_tmp[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d_tmp[key2])         
        dic_df_scale_met_ue_u[key2] = Transformer.transform(dic_df_met_ue_u_tmp[key2]) 
        dic_df_scale_met_ue_d[key2] = Transformer.transform(dic_df_met_ue_d_tmp[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d, dic_predict_met_ue_u, dic_predict_met_ue_d = [{} for _ in range(5)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])        
        dic_predict_met_ue_u[key3] = model.predict(dic_df_scale_met_ue_u[key3], batch_size=dic_df_scale_met_ue_u[key3].shape[0])
        dic_predict_met_ue_d[key3] = model.predict(dic_df_scale_met_ue_d[key3], batch_size=dic_df_scale_met_ue_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d    
    nested_dic_predict_met_ue_u[model_mass] = dic_predict_met_ue_u
    nested_dic_predict_met_ue_d[model_mass] = dic_predict_met_ue_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort, dic_predict_met_ue_u_sort, dic_predict_met_ue_d_sort = [{} for _ in range(5)]

for key4 in dic_df_scale:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()    
    new_df_met_ue_u = pd.DataFrame()
    new_df_met_ue_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weightmetUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weightmetDown"].copy(deep=True).reset_index(drop="True")        
    new_df_met_ue_u["weightmetueUp"] = dic_df_met_ue_u_unsort[key4]["weightmetueUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_ue_d["weightmetueDown"] = dic_df_met_ue_d_unsort[key4]["weightmetueDown"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d    
    dic_predict_met_ue_u_sort[key4] = new_df_met_ue_u
    dic_predict_met_ue_d_sort[key4] = new_df_met_ue_d
    
for model_mass in model_masses:
    for key5 in dic_df_scale:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]        
        dic_predict_met_ue_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_ue_u[model_mass][key5][:,0]
        dic_predict_met_ue_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_ue_d[model_mass][key5][:,0]
    
dic_nested_shape_sys={}
dic_nested_shape_sys["met_u"]=dic_predict_met_u_sort
dic_nested_shape_sys["met_d"]=dic_predict_met_d_sort  
dic_nested_shape_sys["met_ue_u"]=dic_predict_met_ue_u_sort
dic_nested_shape_sys["met_ue_d"]=dic_predict_met_ue_d_sort         
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1688393332.4045653


KeyboardInterrupt: 

2023-07-03 16:08:54.088449: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6fa5c/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-c1d8e/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow/contrib/tensor_fo

# signal unc.

In [ ]:
def write_sys_hist(h1, h2):
    file_name = "Outputs/"+REGION+"/root_files/shape_v1_"+VERSION+model_mass
    f = rt.TFile(file_name+"_v1.root","UPDATE") 
    f.cd("signal_region")
#     h1.Write("zp_"+sys+"Up", rt.TObject.kWriteDelete)
#     h2.Write("zp_"+sys+"Down", rt.TObject.kWriteDelete)
    h1.Write()
    h2.Write()  

In [ ]:
from collections import defaultdict
import math

c = rt.TCanvas()
c.SetGrid()
rt.gStyle.SetOptStat(0)
    
processes = ['TT', 'WW', 'zp']    
# def export_pdf_scale_unc(model_mass): 
for p in processes:
    for model_mass in model_masses:
        if model_mass!= 'Zp-2500_CH-345':
            continue
        if p=='zp':
            key = model_mass
            nset = 31
            score_key = 'score'+model_mass
            str_h = 'zp_pdf'
        else:
            key = p
            nset = 101
            score_key = 'score'+model_mass
            if p == 'TT':
                str_h = 'ttbar_pdf'
            if p == 'WW':
                str_h = 'ww_pdf'
        w0 = dic_predict_sort[key]['weight'].copy(deep=True)
        hists = []
        
        for j in range(nset):
            h = rt.TH1D("", "", nbin, 0, 1)
            for i, v in enumerate(dic_predict_sort[key][score_key]):
                w = w0[i]*dic_predict_sort[key]['pdf'+str(j)+'_pre'][i]
                h.Fill(v, w)
        #     h.Draw("same")
            hists.append(h)
        # print(hists)
    #     c.Draw()
    #     c.SaveAs("Syst/pdfs_"+model_mass+".pdf")

        dic_yields=defaultdict(list)
        for i, h in enumerate(hists):
            for nb in range(nbin):
                dic_yields["pdf"+str(i)].append(h.GetBinContent(nb+1))

        unc_pdfs = []
        for nb in range(nbin):
            detla_sqrs = []
            for i in range(nset-1):
                delta = dic_yields["pdf"+str(i+1)][nb] - dic_yields["pdf0"][nb]
                detla_sqr = delta*delta
                detla_sqrs.append(detla_sqr)
            sum_delta_sqrs = sum(detla_sqrs)
            unc_pdf = math.sqrt(sum_delta_sqrs)
            unc_pdfs.append(unc_pdf)

        hist=HistStack(dic_predict_sort, score_key, 'weight', nbin, 0, 1, "DNN score") 
        c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/normpdf"+model_mass)
        # print(hist.get_yields_by_bin()[model_mass])

        pdf_norm = np.array(hist.get_yields_by_bin()[key])
        # weight
        unc_pdfs1 = np.array(unc_pdfs)*pdf_norm[-1]/dic_yields["pdf0"][-1]

        print('process:', p, "mass:", model_mass, "norm:", pdf_norm[-1], dic_yields["pdf0"][-1], "new:", unc_pdfs1[-1], unc_pdfs[-1])

        pdf_up = pdf_norm + unc_pdfs1
        pdf_down = pdf_norm - unc_pdfs1

        h_pdfup = rt.TH1D(str_h+"Up", str_h+"Up", nbin, 0, 1)
        h_pdfdown = rt.TH1D(str_h+"Down", str_h+"Down", nbin, 0, 1)
        for nb in range(nbin):
            h_pdfup.SetBinContent(nb+1, pdf_up[nb])
            h_pdfdown.SetBinContent(nb+1, pdf_down[nb])
        
        if p == "zp":
            h_scaleUp = rt.TH1D("zp_scaleUp", "zp_scaleUp", nbin, 0, 1)
            h_scaleDown = rt.TH1D("zp_scaleDown", "zp_scaleDown", nbin, 0, 1)
            for i, v in enumerate(dic_predict_sort[model_mass]["score"+model_mass]):
                wup = w0[i]*dic_predict_sort[model_mass]['genweight_u_pre'][i]
                wdn = w0[i]*dic_predict_sort[model_mass]['genweight_d_pre'][i]
                h_scaleUp.Fill(v, wup)
                h_scaleDown.Fill(v, wdn)

            write_sys_hist(h_scaleUp, h_scaleDown)
        write_sys_hist(h_pdfup, h_pdfdown)

